# MOVA Video Factory — Wan2.2 Animate

This notebook runs only the open-source Wan2.2 Animate pipeline. It is restart-safe: the manifest and `.state/wan_jobs.json` are updated after every item, and valid existing video files are never regenerated unless `FORCE_REGENERATE=True`.

Attach a Kaggle Dataset or repository copy containing `presenter/presenter.png` and Blender driving clips before generation. A Hugging Face token with access to the Wan weights may be required; store it as Kaggle secret `HF_TOKEN`.

In [ ]:
# Configuration — edit only this cell for a normal run.
TEST_ONE = 'BAR_SQUAT:standard'
BATCH_COUNT = 1                 # 0 means every eligible asset
RESUME = True
GENERATE_ALL_MISSING = False
FORCE_REGENERATE = False        # leave False to protect approved assets
REPO_URL = 'https://github.com/Gazzamoo111/IG.git'
PROJECT_DIR = '/kaggle/working/IG'
FACTORY_DIR = f'{PROJECT_DIR}/mova-video-factory'
# Update this template only if the Wan checkout/version documents different flags.
WAN_REPO_URL = 'https://github.com/Wan-Video/Wan2.2.git'
WAN_MODEL_ID = 'Wan-AI/Wan2.2-Animate-14B'
WAN_COMMAND_TEMPLATE = ('python {repo_dir}/generate.py --task animate-14B --ckpt_dir {weights_dir} '
                        '--src_root_path {processed} --refert_num 1 --frame_num 241 '
                        '--offload_model True --convert_model_dtype --save_file {output}')
INPUT_FACTORY_DIR = '/kaggle/input/mova-video-factory'  # optional Dataset copy with presenter + driving clips


In [ ]:
!nvidia-smi
import os, shutil, subprocess, sys
assert shutil.which('nvidia-smi'), 'Enable a Kaggle GPU accelerator, then restart.'
os.environ['WAN_REPO_URL'] = WAN_REPO_URL
os.environ['WAN_MODEL_ID'] = WAN_MODEL_ID
os.environ['WAN_COMMAND_TEMPLATE'] = WAN_COMMAND_TEMPLATE
print('GPU visible and Wan configuration exported.')

In [ ]:
# Runtime dependencies; Wan-specific requirements are installed after its checkout.
!pip -q install -U huggingface_hub imageio-ffmpeg
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token: os.environ['HF_TOKEN'] = token
except Exception:
    print('No HF_TOKEN secret found. This is okay only if model access does not require it.')
print('Dependencies ready.')

In [ ]:
# Clone the MOVA repository once, then optionally overlay large private input files from a Kaggle Dataset.
if not os.path.isdir(PROJECT_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, PROJECT_DIR], check=True)
if os.path.isdir(INPUT_FACTORY_DIR):
    shutil.copytree(INPUT_FACTORY_DIR, FACTORY_DIR, dirs_exist_ok=True)
assert os.path.isfile(f'{FACTORY_DIR}/movements.csv'), 'movements.csv is missing from the factory.'
os.chdir(FACTORY_DIR)
print('Factory:', FACTORY_DIR)

In [ ]:
# Clone/install Wan and download open-source model weights. This can take several minutes.
setup = ['python', 'wan/setup_wan.py', '--clone', '--install', '--download-weights',
         '--repo-url', WAN_REPO_URL, '--model-id', WAN_MODEL_ID]
subprocess.run(setup, check=True)
print('Wan runtime and weights are ready.')

In [ ]:
# Validate the presenter and all manifest driving-clip paths before using GPU time.
import csv
with open('movements.csv', newline='', encoding='utf-8-sig') as f: jobs = list(csv.DictReader(f))
assert len(jobs) == 86, f'Expected 86 jobs, found {len(jobs)}'
missing_presenter = sorted({j.get('reference_image_path', 'presenter/presenter.png') for j in jobs if not os.path.isfile(j.get('reference_image_path', 'presenter/presenter.png'))})
missing_driving = [j.get('driving_video_path', '') for j in jobs if not os.path.isfile(j.get('driving_video_path', ''))]
assert not missing_presenter, f'Missing presenter image(s): {missing_presenter}'
assert not missing_driving, f'Missing driving clips ({len(missing_driving)}), e.g. {missing_driving[:3]}'
print(f'Validated {len(jobs)} manifest jobs and their source media.')

In [ ]:
# First test one clip. Run this before a batch whenever Wan's command template changes.
if TEST_ONE:
    command = ['python', 'wan/generate_single.py', '--job-id', TEST_ONE]
    if FORCE_REGENERATE: command.append('--force')
    subprocess.run(command, check=True)
    print('Test job complete:', TEST_ONE)

In [ ]:
# Batch execution. Resume retries failed/running jobs up to three times; normal batch selects planned/missing jobs.
if GENERATE_ALL_MISSING or BATCH_COUNT:
    if RESUME:
        command = ['python', 'wan/resume_batch.py']
    else:
        command = ['python', 'wan/generate_batch.py']
    if not GENERATE_ALL_MISSING: command += ['--count', str(BATCH_COUNT)]
    if FORCE_REGENERATE: command.append('--force')
    subprocess.run(command, check=True)
else:
    print('No batch requested. Set BATCH_COUNT or GENERATE_ALL_MISSING in the configuration cell.')

In [ ]:
# Produce CSV QC and lightweight previews after each batch. Human visual approval is still required.
qc = ['python', 'scripts/qc_media.py', '--manifest', 'movements.csv', '--report', 'output/qc/qc_report.csv']
result = subprocess.run(qc, text=True, capture_output=True)
print(result.stdout or result.stderr)
print('Download /kaggle/working/IG/mova-video-factory/output and output/qc/qc_report.csv from the Kaggle output panel.')